In [2]:
pip install cbbd

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.2/140.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 15.2 MB/s eta 0:00:00
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.3
    Uninstalling pydantic-2.12.3:
      Successfully uninstalled pydantic-2.12.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.11.1 requires pydantic!=2.12.0,!=2.12.1,!=2.4.0,!=2.4.1,<3.0,>=2.0, but you have pydantic 1.10.26 which is incompatible.
pydantic-settings 2.13.1 requires pydantic>=2.7.0, but you have pydantic 1.10.26 which is incompatible.
langchain 1.2.12 requires pydantic<3.0.0,>=2.7.4, but you have pydantic 1.10.26 which is incompatible.
al

In [3]:
#loading data
import cbbd
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

#got this access code from going onto CBBD website- limited by 1000 per month
configuration = cbbd.Configuration(
    access_token = 'jUcAOaOmI7KxZdTYpQM9GNPiM0QAsV7RXipBPgUsYsBIeQ4/bxyKezgt5upprQNc'
)

In [4]:
#loading ncaa tournament games from 2013 to 2025
games = []
with cbbd.ApiClient(configuration) as api_client:
    games_api = cbbd.GamesApi(api_client)
    for season in range(2025, 2013, -1):
        results = games_api.get_games(season=season, tournament='NCAA')
        games += results
len(games)

735

In [5]:
#loading team stats from 2013 to 2025
stats = []
with cbbd.ApiClient(configuration) as api_client:
    stats_api = cbbd.StatsApi(api_client)
    for season in range(2025, 2013, -1):
        results = stats_api.get_team_season_stats(season=season, season_type='regular')
        stats += results
len(stats)

7840

In [6]:
games[:1]

[GameInfo(id=209492, source_id='401745907', season_label='20242025', season=2025, season_type=<SeasonType.POSTSEASON: 'postseason'>, start_date=datetime.datetime(2025, 3, 18, 22, 40, tzinfo=datetime.timezone.utc), start_time_tbd=False, neutral_site=True, conference_game=False, game_type='TRNMNT', tournament='NCAA', game_notes="Men's Basketball Championship - South Region - First Four", status=<GameStatus.FINAL: 'final'>, attendance=0, home_team_id=6, home_team='Alabama State', home_conference_id=25, home_conference='SWAC', home_seed=16, home_points=70, home_period_points=[34, 36], home_winner=True, home_team_elo_start=1618, home_team_elo_end=1625, away_team_id=278, away_team='St. Francis (PA)', away_conference_id=21, away_conference='NEC', away_seed=16, away_points=68, away_period_points=[39, 29], away_winner=False, away_team_elo_start=1735, away_team_elo_end=1728, excitement=7.6, venue_id=76, venue='UD Arena', city='Dayton', state='OH')]

In [7]:
stats[0]

TeamSeasonStats(season=2025, season_label='20242025', team_id=1, team='Abilene Christian', conference='WAC', games=32, wins=16, losses=16, total_minutes=1310, pace=66.4, team_stats=TeamSeasonUnitStats(field_goals=TeamSeasonUnitStatsFieldGoals(pct=44.4, attempted=1845, made=819), two_point_field_goals=TeamSeasonUnitStatsFieldGoals(pct=49.4, attempted=1362, made=673), three_point_field_goals=TeamSeasonUnitStatsFieldGoals(pct=30.2, attempted=483, made=146), free_throws=TeamSeasonUnitStatsFieldGoals(pct=71.5, attempted=666, made=476), rebounds=TeamSeasonUnitStatsRebounds(total=1083, defensive=727, offensive=356), turnovers=TeamSeasonUnitStatsTurnovers(team_total=14, total=476), fouls=TeamSeasonUnitStatsFouls(flagrant=0, technical=9, total=671), points=TeamSeasonUnitStatsPoints(fast_break=359, off_turnovers=581, in_paint=1164, total=2260), four_factors=TeamSeasonUnitStatsFourFactors(free_throw_rate=36.1, offensive_rebound_pct=32.9, turnover_ratio=0.22, effective_field_goal_pct=48.3), assist

In [8]:
# Initial stats to include in model - can experiment to try new things
records = []
for game in games:
    record = game.to_dict()

    # Safeguard against the list being empty
    home_stats_list = [stat for stat in stats if stat.team_id == game.home_team_id and stat.season == game.season]
    away_stats_list = [stat for stat in stats if stat.team_id == game.away_team_id and stat.season == game.season]

    if not home_stats_list or not away_stats_list:
        # If either stat list is empty, skip this game
        continue

    home_stats = home_stats_list[0]
    away_stats = away_stats_list[0]

    try:
        def safe_stat(stat):
            return stat if stat is not None else 0.0  # Replace None with 0.0

        record['home_pace'] = safe_stat(home_stats.pace)
        record['home_o_rating'] = safe_stat(home_stats.team_stats.rating)
        record['home_d_rating'] = safe_stat(home_stats.opponent_stats.rating)
        record['home_free_throw_rate'] = safe_stat(home_stats.team_stats.four_factors.free_throw_rate)
        record['home_turnover_ratio'] = safe_stat(home_stats.team_stats.four_factors.turnover_ratio)
        record['home_efg'] = safe_stat(home_stats.team_stats.four_factors.effective_field_goal_pct)
        record['home_free_throw_rate_allowed'] = safe_stat(home_stats.opponent_stats.four_factors.free_throw_rate)
        record['home_offensive_rebound_rate_allowed'] = safe_stat(home_stats.opponent_stats.four_factors.offensive_rebound_pct)
        record['home_turnover_ratio_forced'] = safe_stat(home_stats.opponent_stats.four_factors.turnover_ratio)
        record['home_efg_allowed'] = safe_stat(home_stats.opponent_stats.four_factors.effective_field_goal_pct)
        record['home_field_goal_pct'] = safe_stat(home_stats.team_stats.field_goals.pct)

        record['away_pace'] = safe_stat(away_stats.pace)
        record['away_o_rating'] = safe_stat(away_stats.team_stats.rating)
        record['away_d_rating'] = safe_stat(away_stats.opponent_stats.rating)
        record['away_free_throw_rate'] = safe_stat(away_stats.team_stats.four_factors.free_throw_rate)
        record['away_turnover_ratio'] = safe_stat(away_stats.team_stats.four_factors.turnover_ratio)
        record['away_efg'] = safe_stat(away_stats.team_stats.four_factors.effective_field_goal_pct)
        record['away_free_throw_rate_allowed'] = safe_stat(away_stats.opponent_stats.four_factors.free_throw_rate)
        record['away_offensive_rebound_rate_allowed'] = safe_stat(away_stats.opponent_stats.four_factors.offensive_rebound_pct)
        record['away_turnover_ratio_forced'] = safe_stat(away_stats.opponent_stats.four_factors.turnover_ratio)
        record['away_efg_allowed'] = safe_stat(away_stats.opponent_stats.four_factors.effective_field_goal_pct)
        record['away_field_goal_pct'] = safe_stat(away_stats.team_stats.field_goals.pct)

        # Differential additions
        record['pace_diff'] = record['home_pace'] - record['away_pace']
        record['rating_diff'] = record['home_o_rating'] - record['away_o_rating']
        record['d_rating_diff'] = record['home_d_rating'] - record['away_d_rating']
        record['efg_diff'] = record['home_efg'] - record['away_efg']
        record['turnover_ratio_diff'] = record['home_turnover_ratio'] - record['away_turnover_ratio']

        # Performance differential feature
        record['home_away_performance_diff'] = (record['home_o_rating'] - record['home_d_rating']) - (record['away_o_rating'] - record['away_d_rating'])

        # Avoid division by zero for win/loss ratio
        record['home_win_loss_ratio'] = home_stats.wins / (home_stats.losses + 1) if home_stats.losses is not None else float('inf')
        record['away_win_loss_ratio'] = away_stats.wins / (away_stats.losses + 1) if away_stats.losses is not None else float('inf')

        records.append(record)

    except AttributeError as e:
        # Handle the case where expected attributes are missing
        print(f"AttributeError: {e} in game {game}")

len(records)

735

In [9]:
#load everything into a data frame
df = pd.DataFrame(records)
df['margin'] = df.homePoints - df.awayPoints
df.head()

,id,sourceId,seasonLabel,season,seasonType,startDate,startTimeTbd,neutralSite,conferenceGame,gameType,...,away_field_goal_pct,pace_diff,rating_diff,d_rating_diff,efg_diff,turnover_ratio_diff,home_away_performance_diff,home_win_loss_ratio,away_win_loss_ratio,margin
0,209492,401745907,20242025,2025,SeasonType.POSTSEASON,2025-03-18 22:40:00+00:00,False,True,False,TRNMNT,...,46.1,5.7,-2.4,-3.9,-5.6,-0.07,1.5,1.187500,0.888889,2
1,209491,401745911,20242025,2025,SeasonType.POSTSEASON,2025-03-19 01:10:00+00:00,False,True,False,TRNMNT,...,47.6,-5.5,-6.2,-8.3,-3.1,0.02,2.1,2.100000,1.571429,-27
2,209493,401745913,20242025,2025,SeasonType.POSTSEASON,2025-03-19 22:40:00+00:00,False,True,False,TRNMNT,...,44.2,-3.7,3.6,2.0,0.7,-0.05,1.6,1.692308,1.692308,-11
3,209573,401745909,20242025,2025,SeasonType.POSTSEASON,2025-03-20 01:18:00+00:00,False,True,False,TRNMNT,...,46.6,-0.9,0.8,2.1,-1.4,-0.02,-1.3,1.187500,1.750000,-6
4,209508,401745959,20242025,2025,SeasonType.POSTSEASON,2025-03-20 16:15:00+00:00,False,True,False,TRNMNT,...,47.5,1.0,3.4,-1.9,-2.4,-0.01,5.3,3.375000,2.181818,-14


In [10]:
print(df.columns)

Index(['id', 'sourceId', 'seasonLabel', 'season', 'seasonType', 'startDate',
       'startTimeTbd', 'neutralSite', 'conferenceGame', 'gameType',
       'tournament', 'gameNotes', 'status', 'attendance', 'homeTeamId',
       'homeTeam', 'homeConferenceId', 'homeConference', 'homeSeed',
       'homePoints', 'homePeriodPoints', 'homeWinner', 'homeTeamEloStart',
       'homeTeamEloEnd', 'awayTeamId', 'awayTeam', 'awayConferenceId',
       'awayConference', 'awaySeed', 'awayPoints', 'awayPeriodPoints',
       'awayWinner', 'awayTeamEloStart', 'awayTeamEloEnd', 'excitement',
       'venueId', 'venue', 'city', 'state', 'home_pace', 'home_o_rating',
       'home_d_rating', 'home_free_throw_rate', 'home_turnover_ratio',
       'home_efg', 'home_free_throw_rate_allowed',
       'home_offensive_rebound_rate_allowed', 'home_turnover_ratio_forced',
       'home_efg_allowed', 'home_field_goal_pct', 'away_pace', 'away_o_rating',
       'away_d_rating', 'away_free_throw_rate', 'away_turnover_ratio',
 

## Train Model

In [11]:
df.columns

Index(['id', 'sourceId', 'seasonLabel', 'season', 'seasonType', 'startDate',
       'startTimeTbd', 'neutralSite', 'conferenceGame', 'gameType',
       'tournament', 'gameNotes', 'status', 'attendance', 'homeTeamId',
       'homeTeam', 'homeConferenceId', 'homeConference', 'homeSeed',
       'homePoints', 'homePeriodPoints', 'homeWinner', 'homeTeamEloStart',
       'homeTeamEloEnd', 'awayTeamId', 'awayTeam', 'awayConferenceId',
       'awayConference', 'awaySeed', 'awayPoints', 'awayPeriodPoints',
       'awayWinner', 'awayTeamEloStart', 'awayTeamEloEnd', 'excitement',
       'venueId', 'venue', 'city', 'state', 'home_pace', 'home_o_rating',
       'home_d_rating', 'home_free_throw_rate', 'home_turnover_ratio',
       'home_efg', 'home_free_throw_rate_allowed',
       'home_offensive_rebound_rate_allowed', 'home_turnover_ratio_forced',
       'home_efg_allowed', 'home_field_goal_pct', 'away_pace', 'away_o_rating',
       'away_d_rating', 'away_free_throw_rate', 'away_turnover_ratio',
 

In [12]:
#pull out the columns we want to use
features = [
    'home_o_rating',
    'home_d_rating',
    'home_pace',
    'home_free_throw_rate',
    #'home_offensive_rebound_rate',
    'home_turnover_ratio',
    'home_efg',
    'home_free_throw_rate_allowed',
    'home_offensive_rebound_rate_allowed',
    'home_turnover_ratio_forced',
    'home_efg_allowed',
    #additions:
    'home_field_goal_pct',
    'away_o_rating',
    'away_d_rating',
    'away_pace',
    'away_free_throw_rate',
    #'away_offensive_rebound_rate',
    'away_turnover_ratio',
    'away_efg',
    'away_free_throw_rate_allowed',
    'away_offensive_rebound_rate_allowed',
    'away_turnover_ratio_forced',
    'away_efg_allowed',
    'homeSeed',
    'awaySeed',
    #additons:
    'away_field_goal_pct',
    #differential additions:
    'pace_diff',
    'rating_diff',
    'd_rating_diff',
    'efg_diff',
    'turnover_ratio_diff',
    'home_away_performance_diff',
    'home_win_loss_ratio',
    'away_win_loss_ratio'
]

outputs = ['margin']

df[features + outputs]

,home_o_rating,home_d_rating,home_pace,home_free_throw_rate,home_turnover_ratio,home_efg,home_free_throw_rate_allowed,home_offensive_rebound_rate_allowed,home_turnover_ratio_forced,home_efg_allowed,...,away_field_goal_pct,pace_diff,rating_diff,d_rating_diff,efg_diff,turnover_ratio_diff,home_away_performance_diff,home_win_loss_ratio,away_win_loss_ratio,margin
0,106.3,104.7,68.3,29.7,0.13,47.6,38.5,27.5,0.19,50.0,...,46.1,5.7,-2.4,-3.9,-5.6,-0.07,1.5,1.187500,0.888889,2
1,108.7,97.8,63.5,31.4,0.17,51.1,36.7,32.1,0.20,44.9,...,47.6,-5.5,-6.2,-8.3,-3.1,0.02,2.1,2.100000,1.571429,-27
2,106.4,104.2,62.2,30.4,0.17,52.1,34.6,26.2,0.19,51.1,...,44.2,-3.7,3.6,2.0,0.7,-0.05,1.6,1.692308,1.692308,-11
3,117.5,108.2,64.4,36.6,0.15,52.5,39.3,31.0,0.17,48.4,...,46.6,-0.9,0.8,2.1,-1.4,-0.02,-1.3,1.187500,1.750000,-6
4,114.8,100.4,67.8,34.5,0.17,53.4,26.8,29.5,0.19,49.2,...,47.5,1.0,3.4,-1.9,-2.4,-0.01,5.3,3.375000,2.181818,-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
730,118.5,101.5,62.2,33.5,0.18,54.5,39.1,31.2,0.19,46.2,...,44.8,0.5,3.3,0.4,3.1,-0.01,2.9,2.888889,2.888889,-6
731,121.4,106.4,60.0,35.9,0.15,55.5,27.6,30.0,0.18,49.6,...,45.1,-5.0,8.2,7.4,5.8,-0.03,0.8,2.777778,2.181818,-3
732,118.4,96.6,59.2,43.4,0.19,52.7,32.6,31.7,0.23,45.1,...,44.8,-2.5,3.2,-4.5,1.3,0.00,7.7,10.666667,2.888889,-10
733,120.4,105.2,61.2,44.1,0.13,53.1,25.9,29.1,0.16,47.5,...,45.1,-3.8,7.2,6.2,3.4,-0.05,1.0,3.250000,2.181818,-1


In [13]:
#split into train/test set using 2025 tournament games as our test data
training = df.query("season != 2025").copy()
testing = df.query("season == 2025").copy()

In [14]:
#further split training data into train/test
X_train, X_valid, y_train, y_valid = train_test_split(training[features], training[outputs], train_size=0.8, test_size=0.2, random_state=0)

## XGBoost

In [15]:
%%capture --no-display
!pip install optuna
import optuna

In [16]:
#XGBoost
def objective(trial):
    param = {
        'tree_method': 'hist',  # Use histogram-based algorithm for faster optimization
        'random_state': 0,
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5)
    }

    XGB_model = XGBRegressor(**param)
    XGB_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)],  verbose=False)

    y_pred = XGB_model.predict(X_valid)
    mae = mean_absolute_error(y_valid, y_pred)
    return mae




In [17]:
# Run the optimization
sampler=optuna.samplers.TPESampler(seed = 42)
study = optuna.create_study(sampler= sampler,direction='minimize')
study.optimize(objective, n_trials=500)

In [18]:
# Print the best hyperparameters
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

Number of finished trials: 500
Best trial: {'n_estimators': 59, 'max_depth': 3, 'learning_rate': 0.035936494583377665, 'subsample': 0.6530766020619997, 'colsample_bytree': 0.9925448757430931, 'gamma': 3.6992721870665703, 'reg_alpha': 1.9246274101266465, 'reg_lambda': 4.235284246905747}


In [19]:
best_params = study.best_trial.params
best_xgb = XGBRegressor(**best_params, random_state=1, verbosity=0)
best_xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9925448757430931, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=3.6992721870665703, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.035936494583377665,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=59, n_jobs=None,
             num_parallel_tree=None, ...)

In [20]:
predictions = best_xgb.predict(X_valid)

mae = mean_absolute_error(y_valid, predictions)

print(mae)
#baseline 8.65

9.279794692993164


In [21]:
predictions = best_xgb.predict(testing[features])
testing['prediction'] = predictions
testing[['homeSeed', 'homeTeam', 'awaySeed', 'awayTeam', 'margin', 'prediction']].head(20)

,homeSeed,homeTeam,awaySeed,awayTeam,margin,prediction
0,16.0,Alabama State,16.0,St. Francis (PA),2,2.453435
1,11.0,San Diego State,11.0,North Carolina,-27,2.057362
2,16.0,American University,16.0,Mount St. Mary's,-11,7.598506
3,11.0,Texas,11.0,Xavier,-6,0.624670
4,8.0,Louisville,9.0,Creighton,-14,2.345195
5,4.0,Purdue,13.0,High Point,12,5.289468
6,3.0,Wisconsin,14.0,Montana,19,12.025866
7,1.0,Houston,16.0,SIU Edwardsville,38,18.978474
8,1.0,Auburn,16.0,Alabama State,20,20.427855
9,5.0,Clemson,12.0,McNeese,-2,6.991807


In [22]:
#percentage of total games model predicted correct in 2025
testing.query("(margin < 0 and prediction < 0) or (margin > 0 and prediction > 0)").shape[0] / testing.shape[0]
#baseline .701

0.7611940298507462

In [23]:
#percentage of first round games model predicted correct in 2025
testing[testing['gameNotes'].str.contains('1st')].query("(margin < 0 and prediction < 0) or (margin > 0 and prediction > 0)").shape[0] / testing[testing['gameNotes'].str.contains('1st')].shape[0]
#baseline .75

0.8125

## Saving Model

In [24]:
import pickle

In [25]:
file_name = "marchmadness.pkl"
pickle.dump(best_xgb, open(file_name, "wb"))

## Predictor for 2026 tournament

In [26]:
import pandas as pd

stats = stats_api.get_team_season_stats(season=2025, season_type='regular')

def predict_game(best_xgb, stats, projected_home_seed, home_team, projected_away_seed, away_team):
    home_stats = [stat for stat in stats if stat.team == home_team][0]
    away_stats = [stat for stat in stats if stat.team == away_team][0]
    record = {
        'home_o_rating': home_stats.team_stats.rating,
        'home_d_rating': home_stats.opponent_stats.rating,
        'home_pace': home_stats.pace,
        'home_free_throw_rate': home_stats.team_stats.four_factors.free_throw_rate,
        #'home_offensive_rebound_rate': home_stats.team_stats.four_factors.offensive_rebound_pct,
        'home_turnover_ratio': home_stats.team_stats.four_factors.turnover_ratio,
        'home_efg': home_stats.team_stats.four_factors.effective_field_goal_pct,
        'home_free_throw_rate_allowed': home_stats.opponent_stats.four_factors.free_throw_rate,
        'home_offensive_rebound_rate_allowed': home_stats.opponent_stats.four_factors.offensive_rebound_pct,
        'home_turnover_ratio_forced': home_stats.opponent_stats.four_factors.turnover_ratio,
        'home_efg_allowed': home_stats.opponent_stats.four_factors.effective_field_goal_pct,
        #additions
        'home_field_goal_pct': home_stats.team_stats.field_goals.pct,
        'away_o_rating': away_stats.team_stats.rating,
        'away_d_rating': away_stats.opponent_stats.rating,
        'away_pace': away_stats.pace,
        'away_free_throw_rate': away_stats.team_stats.four_factors.free_throw_rate,
        #'away_offensive_rebound_rate': away_stats.team_stats.four_factors.offensive_rebound_pct,
        'away_turnover_ratio': away_stats.team_stats.four_factors.turnover_ratio,
        'away_efg': away_stats.team_stats.four_factors.effective_field_goal_pct,
        'away_free_throw_rate_allowed': away_stats.opponent_stats.four_factors.free_throw_rate,
        'away_offensive_rebound_rate_allowed': away_stats.opponent_stats.four_factors.offensive_rebound_pct,
        'away_turnover_ratio_forced': away_stats.opponent_stats.four_factors.turnover_ratio,
        'away_efg_allowed': away_stats.opponent_stats.four_factors.effective_field_goal_pct,
        'homeSeed': projected_home_seed,
        'awaySeed': projected_away_seed,
        #additions:
        'away_field_goal_pct': away_stats.team_stats.field_goals.pct,
         # Differential additions
        'pace_diff': home_stats.pace - away_stats.pace,
        'rating_diff': home_stats.team_stats.rating - away_stats.team_stats.rating,
        'd_rating_diff': home_stats.opponent_stats.rating - away_stats.opponent_stats.rating,
        'efg_diff': home_stats.team_stats.four_factors.effective_field_goal_pct - away_stats.team_stats.four_factors.effective_field_goal_pct,
        'turnover_ratio_diff': home_stats.team_stats.four_factors.turnover_ratio - away_stats.team_stats.four_factors.turnover_ratio,

        # Performance differential feature
        'home_away_performance_diff': (home_stats.team_stats.rating - home_stats.opponent_stats.rating) - (away_stats.team_stats.rating - away_stats.opponent_stats.rating),

        # Avoid division by zero for win/loss ratio
        'home_win_loss_ratio': home_stats.wins / (home_stats.losses + 1) if home_stats.losses is not None else float('inf'),
        'away_win_loss_ratio': away_stats.wins / (away_stats.losses + 1) if away_stats.losses is not None else float('inf')
    }
    return best_xgb.predict(pd.DataFrame([record]))[0]



In [27]:
#if favorite is favored by less than 5 make it an upset


In [28]:

predict_game(best_xgb, stats, 11, 'North Carolina', 1, 'San Diego State')

np.float32(-0.23131704)